<a href="https://colab.research.google.com/github/hmmnyamminji/DL/blob/main/day17_practice3_%EC%96%B4%ED%85%90%EC%85%98_%EB%B6%84%EB%A5%98_%EC%9C%84%EC%B9%98.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 셀프 어텐션 기반 감성 분류 모델

# 어텐션은 집합 연산 — 순서를 모른다
# 위치 임베딩 — 단어에 '자리 번호표'를 더해준다

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import urllib.request, os, math
from collections import Counter  # Counter=단어 빈도 세기

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# 셀 1.데이터 준비  NSMC — 네이버 영화 리뷰
URL = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"
urllib.request.urlretrieve(URL, "ratings_train.txt")
print("ratings_train.txt 다운로드 완료")
texts, labels = [], []
with open("ratings_train.txt", encoding="utf-8") as f:
  next(f) # 첫 줄(헤더 id document label)을 한 번 읽어 버림
  for line in f:
    parts = line.strip().split("\t")
    if len(parts) == 3 and parts[1]:
      texts.append(parts[1])
      labels.append(int(parts[2]))
print(f"전체 리뷰 {len(texts):,}개 | 긍정 {sum(labels):,} / 부정 {len(labels)-sum(labels):,}")
print("샘플:", texts[0], "->", "긍정" if labels[0] else "부정")

N = 30000
texts, labels = texts[:N], labels[:N] # 3만개만 슬라이싱

ratings_train.txt 다운로드 완료
전체 리뷰 149,995개 | 긍정 74,825 / 부정 75,170
샘플: 아 더빙.. 진짜 짜증나네요 목소리 -> 부정


In [ ]:
# 셀 2. vocab — '빈도 상위'만
counter = Counter(tok for t in texts for tok in t.split()) #    {단어: 등장횟수}
print(f"\n고유 어절 수: {len(counter):,}개 ")
print("최다 빈도:", counter.most_common(5))

# 1.   단어사전 생성
VOCAB_SIZE = 15000
vocab = {"<pad>": 0, "<unk>": 1}
for tok, _ in counter.most_common(VOCAB_SIZE - 2): #특수토큰 2개 뺀 만큼
  vocab[tok] = len(vocab)  # 번호 부여, 단어사전 생성

# 2. 문장의 단어를 번호로 바꾸기
MAX_LEN = 20
def encode(text):
  ids = [vocab.get(t, 1) for t in text.split()][:MAX_LEN] #단어를 번호로 바꾸고, 너무 길면 자른다
  return ids + [0] * (MAX_LEN - len(ids))  #너무 짧으면 뒤를 0으로 채운다
print(f"\n 원문: {texts[0]!r}")
print(f"encode(texts[0]) = {encode(texts[0])}") # 첫 리뷰 문장의 단어를 번호로 바꾼거


X = torch.tensor([encode(t) for t in texts])
y = torch.tensor(labels, dtype=torch.float32).reshape(-1, 1)


print(f"\n X.shape = {X.shape}")
print(f"\n y.shape = {y.shape}")

print(f" X[0] = {X[0]}")
print(f" y[0] = {y[0]}")

# 학습/평가 분리
n_train = int(N * 0.9)
train_loader = DataLoader(TensorDataset(X[:n_train], y[:n_train]),batch_size=256, shuffle=True) #  앞 90%  # [256, 20] 문장, 단어
X_test, y_test = X[n_train:].to(device), y[n_train:].to(device) # 뒤 10%

print(f"X_test.shape = {X_test.shape}, y_test.shape = {y_test.shape}")


고유 어절 수: 98,463개 
최다 빈도: [('영화', 2241), ('너무', 1602), ('정말', 1566), ('진짜', 1191), ('이', 1021)]

 원문: '아 더빙.. 진짜 짜증나네요 목소리'
encode(texts[0]) = [51, 1, 5, 10248, 1557, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

 X.shape = torch.Size([30000, 20])

 y.shape = torch.Size([30000, 1])
 X[0] = tensor([   51,     1,     5, 10248,  1557,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0])
 y[0] = tensor([0.])
X_test.shape = torch.Size([3000, 20]), y_test.shape = torch.Size([3000, 1])


In [ ]:
# 셀 2. 모델 — 어텐션 분류기 (위치 임베딩 스위치 포함)
class AttnSentiment(nn.Module):
  def __init__(self, use_position):
    super().__init__()
    self.use_position = use_position

    self.emb = nn.Embedding(len(vocab), 64, padding_idx=0)  #(15000, 64)
    self.pos = nn.Embedding(MAX_LEN, 64) #(20, 64) 각 단어마다 위치 벡터가 생성됨 # 학습형 위치 임베딩(GPT-2) 모델이 학습.원조 Transformer는 고정형 위치 임베딩 사용.

    self.W_q = nn.Linear(64, 64, bias=False) #셀프 어텐션의 Q·K·V 를 만드는 학습 행렬 3개
    self.W_k = nn.Linear(64, 64, bias=False) #학습 행렬은 모델이 데이터로부터 배우면서 Q, K, V 벡터를 만들어내는 방법을 결정하는 조정 가능한 파라미터
    self.W_v = nn.Linear(64, 64, bias=False) #입력 벡터를 '질문(Query),열쇠(Key),  값(Value)' 벡터로 변환

    self.fc = nn.Sequential(nn.Linear(64, 32), nn.ReLU(),nn.Linear(32, 1), nn.Sigmoid()) # 0~1 확률값 (긍정/부정)

  def forward(self, x): # x: (B, 20)
    h = self.emb(x)  #(B, 20, 64)
    if self.use_position:
      positions = torch.arange(x.size(1), device=x.device) # 0번째 단어부터 19번째 단어까지 각 단어의 위치 임베딩(자리 번호 생성)
      h = h + self.pos(positions) #  (B, 20, 64) + (20,64)
    Q, K, V = self.W_q(h), self.W_k(h), self.W_v(h) # (B,20,64)
    scores = Q @ K.transpose(1, 2) / math.sqrt(64) # (B,20,20) 모든 단어쌍 관련도
    attn = F.softmax(scores, dim=-1)  # 각 행(질문 단어)마다 무게 합=1 , [(B,20,20)]
    h = attn @ V # 무게 × 내용 → (B, 20, 64)

    # 문장 벡터 = 유효 단어들의 평균
    mask = (x != 0).unsqueeze(-1).float()  #(B,20,1)
    sent = (h * mask).sum(1) / mask.sum(1).clamp(min=1) # 진짜 단어들만 평균 (B,64)
    return self.fc(sent)  # 분류기 → 긍정 확률 (B,1)



In [ ]:
def run(use_position, epochs=4): #학습·평가하는 함수
    torch.manual_seed(42)
    model = AttnSentiment(use_position).to(device)
    loss_fn = nn.BCELoss()
    opt = torch.optim.Adam(model.parameters(), lr=0.002)

    for _ in range(epochs):
      model.train()
      for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        loss = loss_fn(model(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()

    model.eval()
    with torch.no_grad():
      acc = ((model(X_test) > 0.5) == y_test.bool()).float().mean().item()
    return model, acc

def predict(model, sentence): # 문장 하나
  model.eval()
  with torch.no_grad():
    return model(torch.tensor([encode(sentence)]).to(device)).item()


In [ ]:
# 셀 3. 위치 임베딩: 단어에 자리 번호표를
model_pos, acc_pos = run(use_position=True)
print(f"\n[어텐션 + 위치 임베딩] 테스트 정확도 {acc_pos:.4f}")


[어텐션 + 위치 임베딩] 테스트 정확도 0.7137


In [ ]:
a, b = "지루하다 하지만 결말은 최고", "최고 하지만 결말은 지루하다"
pa, pb = predict(model_pos, a), predict(model_pos, b)
print(f"  {a!r}: {pa:.0%}")
print(f"  {b!r}: {pb:.0%}")

  '지루하다 하지만 결말은 최고': 77%
  '최고 하지만 결말은 지루하다': 2%


In [ ]:
# 위치 임베딩 '없이' 어텐션만으로 학습·평가
model_nopos, acc_nopos = run(use_position=False)
print(f"[어텐션, 위치 없음] 테스트 정확도 {acc_nopos:.4f}")

[어텐션, 위치 없음] 테스트 정확도 0.7120


In [ ]:
pa, pb = predict(model_nopos, a), predict(model_nopos, b)
print(f"  {a!r}: {pa:.0%}")
print(f"  {b!r}: {pb:.0%}")

  '지루하다 하지만 결말은 최고': 7%
  '최고 하지만 결말은 지루하다': 7%
